# 02 · CQI Feature Engineering

**Environment**: AWS EMR (PySpark)  
**Input**: `s3://se-data-dump-2025/parquet/stackoverflow/Posts` + `PostLinks`  
**Output**: `s3://se-data-dump-2025/features/posts_cqi/`

---

## CQI Dimension Design

| Dimension | Weight | Tool | Python Questions | Non-Python Questions |
|---|---|---|---|---|
| Community Recognition | 0.25 | pandas | norm(log(1+Score)) | norm(log(1+Score)) |
| Code Complexity | 0.20 | Radon cc_visit() | cyclomatic_complexity | **None** |
| Halstead Volume | 0.15 | Radon h_visit() | log(1+halstead_volume) | **None** |
| Code Parseability | 0.10 | ast.parse() | 1.0 / 0.0 | **None** |
| Answerability | 0.10 | pandas | norm(log(1+AnswerCount)) | norm(log(1+AnswerCount)) |
| Not Closed | 0.10 | pandas | 1.0 / 0.0 | 1.0 / 0.0 |
| Not Duplicate | 0.10 | PostLinks JOIN | 1.0 / 0.0 | 1.0 / 0.0 |

**cqi_version_a**: computed only for Python questions; `None` for all others.  
**Parse failure handling**: when cyclomatic_complexity or halstead_volume is None, those dimensions contribute 0. CQI is computed from the remaining available dimensions (weights sum to at least 0.65 without renormalization).

---

## Execution Plan

```
Phase 1: Small sample validation (5,000 rows, ~5-10 min)
Phase 2: Full dataset processing (~2.4M rows, ~40-60 min)
Phase 3: Monthly aggregation — generate CSV for teammate's RDD analysis
Phase 4: S3 output verification
```

---
## 0 · Configuration

In [ ]:
BUCKET         = "se-data-dump-2025"
PARQUET_PREFIX = "parquet/stackoverflow"
FEATURE_PREFIX = "features"
OUTPUT_PREFIX  = "outputs"

POSTS_PATH      = f"s3://{BUCKET}/{PARQUET_PREFIX}/Posts"
POSTLINKS_PATH  = f"s3://{BUCKET}/{PARQUET_PREFIX}/PostLinks"
CQI_SAMPLE_PATH = f"s3://{BUCKET}/{FEATURE_PREFIX}/posts_cqi_sample/"
CQI_FULL_PATH   = f"s3://{BUCKET}/{FEATURE_PREFIX}/posts_cqi/"

# Study time window
WINDOW_START = "2019-01"
WINDOW_END   = "2025-03"

print(f"Posts     : {POSTS_PATH}")
print(f"PostLinks : {POSTLINKS_PATH}")
print(f"CQI output: {CQI_FULL_PATH}")

---
## 1 · Verify Spark Environment

In [ ]:
import warnings
warnings.filterwarnings('ignore')

print("Spark version:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

# Performance tuning
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.parquet.compression.codec", "snappy")

print("Spark config OK.")

---
## 2 · Upload and Distribute the CQI Feature Module

**Prerequisite**: `cqi_features_version_a.py` must be in the same directory as this notebook, or already on S3.  
Verify with: `!ls cqi_features_version_a.py`

In [ ]:
import subprocess
import boto3

# Verify local file exists
result = subprocess.run(["ls", "-lh", "cqi_features_version_a.py"],
                        capture_output=True, text=True)
print(result.stdout or result.stderr)

# Upload to S3
s3_client = boto3.client("s3")
s3_client.upload_file(
    "cqi_features_version_a.py",
    BUCKET,
    "code/cqi_features_version_a.py"
)
print("Uploaded to S3.")

# Distribute to all Executors (required for pandas_udf imports)
spark.sparkContext.addPyFile(
    f"s3://{BUCKET}/code/cqi_features_version_a.py"
)
print("Module distributed to all Executors.")

In [ ]:
# Verify import works correctly
from cqi_features_version_a import compute_python_code_features

test_html = """
<p>How to sort a list in Python?</p>
<pre><code>
def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr
</code></pre>
"""

test_result = compute_python_code_features(test_html)
print("Module test result:")
for k, v in test_result.items():
    print(f"  {k:<28} {v}")

---
## 3 · Define pandas_udf and Helper Functions

In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, when, lit, log1p, greatest, avg, count, mean,
    pandas_udf, date_format
)
from pyspark.sql.types import (
    StructType, StructField, FloatType, IntegerType, StringType
)

# Output schema for pandas_udf
CODE_FEATURE_SCHEMA = StructType([
    StructField("code_block_count",      IntegerType(), True),
    StructField("primary_code_len",      IntegerType(), True),
    StructField("cyclomatic_complexity", FloatType(),   True),
    StructField("halstead_volume",       FloatType(),   True),
    StructField("is_parseable",          FloatType(),   True),
    StructField("has_code",              IntegerType(), True),
])

@pandas_udf(CODE_FEATURE_SCHEMA)
def extract_python_features_udf(body_series: pd.Series) -> pd.DataFrame:
    """
    Extract code features from the HTML Body of Python questions.
    Called only on rows where is_python == 1.
    """
    from cqi_features_version_a import compute_python_code_features
    results = body_series.apply(compute_python_code_features)
    return pd.DataFrame(results.tolist())


def add_is_python(df):
    """Flag rows as Python-related questions based on Tags."""
    return df.withColumn(
        "is_python",
        when(
            col("Tags").contains("<python>") |
            col("Tags").contains("<python-3") |
            col("Tags").contains("<python-2"),
            1
        ).otherwise(0)
    )


def add_is_duplicate(df, postlinks_df):
    """Join PostLinks to flag duplicate questions (LinkTypeId = 3)."""
    duplicates = postlinks_df \
        .filter(col("LinkTypeId") == "3") \
        .select(col("PostId").alias("Id")) \
        .distinct() \
        .withColumn("is_duplicate", lit(1))

    return df.join(duplicates, on="Id", how="left") \
             .fillna({"is_duplicate": 0})


def add_code_features(df):
    """
    Extract code features for Python questions via pandas_udf.
    Non-Python questions get None for all code-related columns.
    """
    # Apply UDF only to Python questions
    df = df.withColumn(
        "_code_features",
        when(
            col("is_python") == 1,
            extract_python_features_udf(col("Body"))
        ).otherwise(None)
    )

    # Expand struct column into individual columns
    for field in CODE_FEATURE_SCHEMA.fields:
        df = df.withColumn(
            field.name,
            col(f"_code_features.{field.name}")
        )

    # Drop intermediate struct and raw Body (no longer needed)
    return df.drop("_code_features", "Body")


def compute_cqi_version_a(df):
    """
    Compute CQI Version A for Python questions only.

    Weights:
      Community recognition  0.25  log(1+Score)
      Code complexity        0.20  cyclomatic_complexity (None -> 0)
      Halstead volume        0.15  log(1+halstead_volume) (None -> 0)
      Code parseability      0.10  is_parseable
      Answerability          0.10  log(1+AnswerCount)
      Not closed             0.10  ClosedDate is null
      Not duplicate          0.10  is_duplicate == 0

    Non-Python questions: cqi_version_a = None.
    """
    # Intermediate dimensions
    df = df.withColumn("_recognition",
        log1p(greatest(col("Score").cast("float"), lit(0.0))))

    df = df.withColumn("_answerability",
        log1p(col("AnswerCount").cast("float")))

    df = df.withColumn("_not_closed",
        when(col("ClosedDate").isNull(), 1.0).otherwise(0.0))

    df = df.withColumn("_not_duplicate",
        when(col("is_duplicate") == 0, 1.0).otherwise(0.0))

    # CQI: Python questions only
    df = df.withColumn(
        "cqi_version_a",
        when(
            col("is_python") == 1,
            col("_recognition")   * 0.25 +
            col("_answerability") * 0.10 +
            col("_not_closed")    * 0.10 +
            col("_not_duplicate") * 0.10 +
            when(col("cyclomatic_complexity").isNotNull(),
                 col("cyclomatic_complexity") * 0.20
            ).otherwise(0.0) +
            when(col("halstead_volume").isNotNull(),
                 log1p(col("halstead_volume")) * 0.15
            ).otherwise(0.0) +
            when(col("is_parseable").isNotNull(),
                 col("is_parseable") * 0.10
            ).otherwise(0.0)
        ).otherwise(None)
    )

    # Drop intermediate columns
    return df.drop("_recognition", "_answerability",
                   "_not_closed", "_not_duplicate")


print("All functions defined.")

---
## 4 · Load Raw Data

In [ ]:
import time as _time

print("Loading Posts Parquet ...")
posts_raw = spark.read.parquet(POSTS_PATH)
print(f"  columns: {posts_raw.columns}")

print("\nLoading PostLinks Parquet ...")
postlinks_raw = spark.read.parquet(POSTLINKS_PATH)
print(f"  columns: {postlinks_raw.columns}")

print("\nData loaded (lazy evaluation — actual reads triggered on action).")

---
## Phase 1 · Small Sample Validation

Use 5,000 rows from Oct–Nov 2022 to run the full pipeline end-to-end.  
Only proceed to Phase 2 after all checks pass.

In [ ]:
print("[Phase 1] Building small sample ...")
t0 = _time.time()

sample_raw = posts_raw.filter(
    (col("year_month") >= "2022-10") &
    (col("year_month") <= "2022-11")
).limit(5000)

print(f"  Sample row count: {sample_raw.count()}")
sample_raw.show(3, truncate=80)
print(f"  Elapsed: {(_time.time()-t0):.1f}s")

In [ ]:
print("[Phase 1] Step 1/4 — Flagging is_python ...")
sample = add_is_python(sample_raw)

python_count = sample.filter(col("is_python") == 1).count()
total_count  = sample.count()
print(f"  Python questions: {python_count} / {total_count} "
      f"({python_count/total_count*100:.1f}%)")

In [ ]:
print("[Phase 1] Step 2/4 — Joining PostLinks, flagging is_duplicate ...")
sample = add_is_duplicate(sample, postlinks_raw)

dup_count = sample.filter(col("is_duplicate") == 1).count()
print(f"  Duplicate questions: {dup_count} / {total_count} "
      f"({dup_count/total_count*100:.1f}%)")

In [ ]:
print("[Phase 1] Step 3/4 — Extracting code features (pandas_udf) ...")
print("  This triggers actual computation. Expect ~2-5 min for 5k rows ...", flush=True)
t0 = _time.time()

sample = add_code_features(sample)

code_cols = ["Id", "is_python", "code_block_count", "primary_code_len",
             "cyclomatic_complexity", "halstead_volume", "is_parseable", "has_code"]
sample.select(code_cols).show(10)

print(f"  Elapsed: {(_time.time()-t0)/60:.1f} min")

In [ ]:
print("[Phase 1] Step 4/4 — Computing CQI Version A ...")
sample = compute_cqi_version_a(sample)

sample.select("Id", "is_python", "Score", "AnswerCount",
              "cyclomatic_complexity", "halstead_volume",
              "is_parseable", "cqi_version_a").show(10)

print("\nCQI distribution (Python questions only):")
sample.filter(col("is_python") == 1) \
      .select("cqi_version_a") \
      .summary("count", "mean", "min", "25%", "75%", "max") \
      .show()

In [ ]:
print("[Phase 1] Validation checks ...")

# 1. Code feature availability for Python questions
py_sample = sample.filter(col("is_python") == 1)
print("\nPython questions — code feature coverage:")
py_sample.select(
    avg("has_code").alias("has_code_rate"),
    avg("is_parseable").alias("parse_rate"),
    avg(when(col("cyclomatic_complexity").isNotNull(), 1).otherwise(0)).alias("cc_available_rate"),
    avg(when(col("halstead_volume").isNotNull(), 1).otherwise(0)).alias("hv_available_rate"),
).show()

# 2. Non-Python questions must have None for all code dimensions
print("Non-Python questions — code dimensions should all be None:")
non_py = sample.filter(col("is_python") == 0)
none_check = non_py.select(
    avg(when(col("cyclomatic_complexity").isNotNull(), 1).otherwise(0)).alias("cc_not_null_rate"),
    avg(when(col("halstead_volume").isNotNull(), 1).otherwise(0)).alias("hv_not_null_rate"),
    avg(when(col("is_parseable").isNotNull(), 1).otherwise(0)).alias("parseable_not_null_rate"),
    avg(when(col("cqi_version_a").isNotNull(), 1).otherwise(0)).alias("cqi_not_null_rate"),
).collect()[0]
print(f"  cc_not_null_rate       : {none_check.cc_not_null_rate}  (expected 0.0)")
print(f"  hv_not_null_rate       : {none_check.hv_not_null_rate}  (expected 0.0)")
print(f"  parseable_not_null_rate: {none_check.parseable_not_null_rate}  (expected 0.0)")
print(f"  cqi_not_null_rate      : {none_check.cqi_not_null_rate}  (expected 0.0)")

print("\n[Phase 1] Small sample validation complete.")

In [ ]:
print("[Phase 1] Writing sample results to S3 ...")
t0 = _time.time()

sample.write \
    .mode("overwrite") \
    .parquet(CQI_SAMPLE_PATH)

print(f"  Written to: {CQI_SAMPLE_PATH}")
print(f"  Elapsed: {(_time.time()-t0)/60:.1f} min")

# Verify readable
verify = spark.read.parquet(CQI_SAMPLE_PATH)
print(f"  Verified row count : {verify.count():,}")
print(f"  Column count       : {len(verify.columns)}")

---
## Phase 2 · Full Dataset Processing

⚠️ **Only run this section after Phase 1 passes all checks.**  
Full dataset: ~2.4M Python questions. Expected runtime: 40–60 minutes.

In [ ]:
print("[Phase 2] Starting full dataset processing ...", flush=True)
t_total = _time.time()

# Step 1: Flag is_python
print("\n  [1/4] Flagging is_python ...", flush=True)
posts_full = add_is_python(posts_raw)

# Step 2: Join PostLinks
print("  [2/4] Joining PostLinks, flagging is_duplicate ...", flush=True)
t0 = _time.time()
posts_full = add_is_duplicate(posts_full, postlinks_raw)
print(f"        Done ({(_time.time()-t0)/60:.1f} min)", flush=True)

# Step 3: Code feature extraction
print("  [3/4] Extracting code features (pandas_udf) ...", flush=True)
print("        This is the longest step — expect 30-50 minutes ...", flush=True)
t0 = _time.time()
posts_full = add_code_features(posts_full)
print(f"        Done ({(_time.time()-t0)/60:.1f} min)", flush=True)

# Step 4: Compute CQI
print("  [4/4] Computing CQI Version A ...", flush=True)
posts_full = compute_cqi_version_a(posts_full)
print("        Done.", flush=True)

In [ ]:
print("[Phase 2] Writing full CQI results to S3 ...", flush=True)
t0 = _time.time()

posts_full.write \
    .mode("overwrite") \
    .partitionBy("year_month") \
    .parquet(CQI_FULL_PATH)

elapsed       = (_time.time() - t0) / 60
total_elapsed = (_time.time() - t_total) / 60
print(f"  Written to : {CQI_FULL_PATH}")
print(f"  Write time : {elapsed:.1f} min")
print(f"  Total time : {total_elapsed:.1f} min")

In [ ]:
print("[Phase 2] Full dataset validation ...")
posts_cqi = spark.read.parquet(CQI_FULL_PATH)

total_rows  = posts_cqi.count()
python_rows = posts_cqi.filter(col("is_python") == 1).count()
cqi_rows    = posts_cqi.filter(col("cqi_version_a").isNotNull()).count()

print(f"  Total rows          : {total_rows:,}")
print(f"  Python questions    : {python_rows:,}")
print(f"  Rows with CQI       : {cqi_rows:,}")

print("\nCQI Version A distribution (Python questions):")
posts_cqi.filter(col("is_python") == 1) \
         .select("cqi_version_a") \
         .summary("count", "mean", "min", "25%", "75%", "max") \
         .show()

print("\nCode feature availability (Python questions):")
posts_cqi.filter(col("is_python") == 1).select(
    avg("has_code").alias("has_code_rate"),
    avg("is_parseable").alias("parse_rate"),
    avg(when(col("cyclomatic_complexity").isNotNull(), 1)
        .otherwise(0)).alias("cc_available"),
    avg(when(col("halstead_volume").isNotNull(), 1)
        .otherwise(0)).alias("hv_available"),
).show()

print("[Phase 2] Full dataset processing complete.")

---
## Phase 3 · Monthly Aggregation

Generate the monthly aggregation CSV for the teammate's RDD regression analysis.

In [ ]:
print("[Phase 3] Building monthly aggregation table ...", flush=True)

posts_cqi = spark.read.parquet(CQI_FULL_PATH)

monthly = posts_cqi \
    .filter(
        (col("is_python") == 1) &
        (col("year_month") >= WINDOW_START) &
        (col("year_month") <= WINDOW_END)
    ) \
    .groupBy("year_month") \
    .agg(
        count("Id").alias("n_questions"),
        avg("cqi_version_a").alias("mean_cqi"),
        avg(col("Score").cast("float")).alias("avg_score"),
        avg(col("AnswerCount").cast("float")).alias("avg_answer_count"),
        mean(when(col("ClosedDate").isNotNull(), 1)
             .otherwise(0).cast("float")).alias("closed_rate"),
        avg(col("is_duplicate").cast("float")).alias("duplicate_rate"),
        avg("has_code").alias("has_code_rate"),
        avg("is_parseable").alias("parse_rate"),
        avg("cyclomatic_complexity").alias("avg_cyclomatic"),
        avg("halstead_volume").alias("avg_halstead"),
    ) \
    .orderBy("year_month")

print(f"  Monthly data: {monthly.count()} months")
monthly.show(10)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

monthly_pd = monthly.toPandas()
monthly_pd["year_month"] = pd.to_datetime(monthly_pd["year_month"])
monthly_pd = monthly_pd.sort_values("year_month")

CUTOFF = pd.Timestamp("2022-11-30")

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: Question volume
axes[0].plot(monthly_pd["year_month"], monthly_pd["n_questions"],
             color="steelblue", linewidth=1.5)
axes[0].axvline(CUTOFF, color="red", linestyle="--", linewidth=1.2,
                label="ChatGPT launch (Nov 2022)")
axes[0].set_ylabel("Question Count")
axes[0].set_title("Stack Overflow Python Questions — Monthly Trends (2019-2025)")
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[0].legend()

# Panel 2: Mean CQI
axes[1].plot(monthly_pd["year_month"], monthly_pd["mean_cqi"],
             color="darkorange", linewidth=1.5)
axes[1].axvline(CUTOFF, color="red", linestyle="--", linewidth=1.2)
axes[1].set_ylabel("Mean CQI (Version A)")

# Panel 3: Average score
axes[2].plot(monthly_pd["year_month"], monthly_pd["avg_score"],
             color="green", linewidth=1.5)
axes[2].axvline(CUTOFF, color="red", linestyle="--", linewidth=1.2)
axes[2].set_ylabel("Avg Score")
axes[2].set_xlabel("Month")

plt.tight_layout()
plt.savefig("monthly_cqi_trends.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved: monthly_cqi_trends.png")

In [ ]:
import boto3

# Save CSV locally
monthly_pd.to_csv("monthly_agg.csv", index=False)
print("CSV saved locally: monthly_agg.csv")

# Upload to S3
s3 = boto3.client("s3")
s3.upload_file("monthly_agg.csv", BUCKET, f"{OUTPUT_PREFIX}/monthly_agg.csv")
print(f"CSV uploaded: s3://{BUCKET}/{OUTPUT_PREFIX}/monthly_agg.csv")

s3.upload_file("monthly_cqi_trends.png", BUCKET, f"{OUTPUT_PREFIX}/monthly_cqi_trends.png")
print(f"Plot uploaded: s3://{BUCKET}/{OUTPUT_PREFIX}/monthly_cqi_trends.png")

print("\nMonthly aggregation preview (first 5 rows):")
print(monthly_pd.head())

---
## Phase 4 · S3 Output Summary

In [ ]:
import subprocess

print("=" * 60)
print("S3 Output Summary")
print("=" * 60)

paths = [
    f"s3://{BUCKET}/{FEATURE_PREFIX}/posts_cqi/",
    f"s3://{BUCKET}/{OUTPUT_PREFIX}/",
]

for path in paths:
    print(f"\n{path}")
    result = subprocess.run(
        ["aws", "s3", "ls", path, "--human-readable", "--summarize"],
        capture_output=True, text=True
    )
    lines = result.stdout.strip().split("\n")
    for line in lines[-3:]:
        print(f"  {line}")

print("\n" + "=" * 60)
print("Next step: push monthly_agg.csv to GitHub for teammate's RDD analysis.")
print("=" * 60)